<a href="https://colab.research.google.com/github/BhagwatiOracle/KrishiVani/blob/main/Finetuning_VIT_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.3/260.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 109.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92


In [ ]:
from datasets import load_dataset

dataset1 = load_dataset("sharmin3/Rice-Leaf-Disease")

Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
dataset1 = dataset1['train'].shuffle().select(range(2000))

In [ ]:
from datasets import load_dataset
dataset2 = load_dataset("Prachi1234/corn-leaf-disease")

In [ ]:
dataset2 = dataset2['train'].shuffle().select(range(2000))

In [ ]:


from roboflow import Roboflow
rf = Roboflow(api_key="LLDhy4hz6Agj4IdZqnPu")
project = rf.workspace("wheat-disease-ntkxi").project("wheat-disease-classification-82mlk")
version = project.version(1)
dataset = version.download("folder")


loading Roboflow workspace...
loading Roboflow project...


In [ ]:
dataset3 = load_dataset("/content/Wheat-Disease-Classification-1",split='train')

Resolving data files:   0%|          | 0/1187 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/139 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/149 [00:00<?, ?it/s]

# **Merging Dataset**

In [ ]:
from datasets import ClassLabel, concatenate_datasets

# Extract string names from each dataset's label feature
labels_ds1 = dataset1.features["label"].names
labels_ds2 = dataset2.features["label"].names
labels_ds3 = dataset3.features["label"].names

# Merge into a single unique sorted list
unified_names = sorted(list(set(labels_ds1 + labels_ds2 + labels_ds3)))

# Create the new master ClassLabel feature object
unified_class_label = ClassLabel(names=unified_names)

In [ ]:
def remap_ds1(example):
    # Convert old ID to string name, then convert name to new unified ID
    string_name = dataset1.features["label"].int2str(example["label"])
    example["label"] = unified_class_label.str2int(string_name)
    return example

def remap_ds2(example):
    # Convert old ID to string name, then convert name to new unified ID
    string_name = dataset2.features["label"].int2str(example["label"])
    example["label"] = unified_class_label.str2int(string_name)
    return example

def remap_ds3(example):
    # Convert old ID to string name, then convert name to new unified ID
    string_name = dataset3.features["label"].int2str(example["label"])
    example["label"] = unified_class_label.str2int(string_name)
    return example

In [ ]:
from datasets import Features, Image

# Define the new features for ds1
new_features_ds1 = Features({
    "image": dataset1.features["image"],
    "label": unified_class_label
})

# Define the new features for ds2
new_features_ds2 = Features({
    "image": dataset2.features["image"],
    "label": unified_class_label
})

# Define the new features for ds3
new_features_ds3 = Features({
    "image": dataset3.features["image"],
    "label": unified_class_label
})

ds1 = dataset1.map(remap_ds1, features=new_features_ds1)
ds2 = dataset2.map(remap_ds2, features=new_features_ds2)
ds3 = dataset3.map(remap_ds3, features=new_features_ds3)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1187 [00:00<?, ? examples/s]

In [ ]:
ds1 = ds1.cast_column("label",unified_class_label)
ds2 = ds2.cast_column("label",unified_class_label)
ds3 = ds3.cast_column("label",unified_class_label)

Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1187 [00:00<?, ? examples/s]

In [ ]:
from datasets import concatenate_datasets

final_dataset = concatenate_datasets([ds1, ds2, ds3])

In [ ]:
final_dataset.features['label']

ClassLabel(names=['Bacterialblight', 'BlackPoint', 'Blast', 'Blight', 'Brownspot', 'Common_Rust', 'FusariumFootRot', 'Gray_Leaf_Spot', 'Healthy', 'HealthyLeaf', 'LeafBlight', 'Tungro', 'WheatBlast'])

In [ ]:
final_dataset

Dataset({
    features: ['image', 'label'],
    num_rows: 5187
})

# **Train-Test Split**

In [ ]:
# Use the train_test_split method of the dataset object
dataset = final_dataset.train_test_split(test_size=0.2, shuffle=True)


In [ ]:
dataset['train'].features['label']

ClassLabel(names=['Bacterialblight', 'BlackPoint', 'Blast', 'Blight', 'Brownspot', 'Common_Rust', 'FusariumFootRot', 'Gray_Leaf_Spot', 'Healthy', 'HealthyLeaf', 'LeafBlight', 'Tungro', 'WheatBlast'])

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 4149
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1038
    })
})

# **Data Preprocessing**

1. Label Mapping

In [ ]:
id2label = {id:label for id, label in enumerate(dataset['train'].features['label'].names)}
label2id = {label:id for id,label in id2label.items()}

In [ ]:
id2label


{0: 'Bacterialblight',
 1: 'BlackPoint',
 2: 'Blast',
 3: 'Blight',
 4: 'Brownspot',
 5: 'Common_Rust',
 6: 'FusariumFootRot',
 7: 'Gray_Leaf_Spot',
 8: 'Healthy',
 9: 'HealthyLeaf',
 10: 'LeafBlight',
 11: 'Tungro',
 12: 'WheatBlast'}

2. Image Processing

In [ ]:
from transformers import ViTImageProcessor

model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

3. Image Augmentation

In [ ]:
from torchvision.transforms import CenterCrop, Compose, Normalize, RandomHorizontalFlip, RandomResizedCrop, ToTensor, Resize

image_mean, image_std = processor.image_mean, processor.image_std
size = processor.size['height']

normalize = Normalize(mean=image_mean, std=image_std)

train_transforms = Compose([
    RandomResizedCrop(size),
    RandomHorizontalFlip(),
    ToTensor(),
    normalize,
])

val_transform = Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    normalize,

])

test_transforms =  Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    normalize,
])

In [ ]:
def apply_train_transforms(examples):
  examples['pixel_values'] = [train_transforms(image.convert("RGB")) for image in examples['image']]
  return examples

def apply_val_transforms(examples):
  examples['pixel_values'] = [val_transform(image.convert("RGB")) for image in examples['image']]
  return examples

def apply_test_transforms(examples):
  examples['pixel_values'] = [test_transforms(image.convert("RGB")) for image in examples['image']]
  return examples

In [ ]:
train_ds = dataset['train']
test_ds = dataset['test']

In [ ]:
train_ds.set_transform(apply_train_transforms)
test_ds.set_transform(apply_test_transforms)

4. Data Loading

In [ ]:
import torch
from torch.utils.data import DataLoader

def collate_fn(examples):
  pixel_values = torch.stack([example['pixel_values'] for example in examples])
  labels = torch.tensor([example['label'] for example in examples])
  return {'pixel_values': pixel_values, 'labels': labels}

train_dl = DataLoader(train_ds, collate_fn=collate_fn, batch_size=8, shuffle=True)

In [ ]:
batch = next(iter(train_dl))
for k,v in batch.items():
  if isinstance(v, torch.Tensor):
    print(k, v.shape)

pixel_values torch.Size([8, 3, 224, 224])
labels torch.Size([8])


# **Fine-Tuning**

In [ ]:
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(model_name, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([13])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([13, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np

train_args = TrainingArguments(
    output_dir = "output-models",
    save_total_limit=2,
    report_to="wandb",
    save_strategy="epoch",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=10,
    per_device_eval_batch_size=4,
    num_train_epochs=40,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir='logs',
    remove_unused_columns=False,
)

In [ ]:
trainer = Trainer(
    model,
    train_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collate_fn,
    tokenizer=processor,
)
trainer.train()